# Example code of TrOCR
Colab 환경에서 TrOCR 모델을 사용해 손글씨를 인식해보는 예제입니다.

## 패키지 및 예제 데이터 다운로드하기
예제를 실행시키기 위해 python package들을 설치합니다. 예제로 사용할 이미지들도 다운로드 받습니다. Colab에서 실행하지 않는 경우 이 셀은 실행하지 않습니다.

In [3]:
!curl -O https://raw.githubusercontent.com/mrsyee/dl_apps/main/ocr/requirements-colab.txt
!pip install -r requirements-colab.txt

# 예제 이미지 다운로드
!mkdir examples
!cd examples && wget https://github.com/mrsyee/dl_apps/raw/main/ocr/examples/Hello.png
!cd examples && wget https://github.com/mrsyee/dl_apps/raw/main/ocr/examples/sentence.png

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100   123  100   123    0     0   2155      0 --:--:-- --:--:-- --:--:--  2236


  Using cached sentencepiece-0.1.97.tar.gz (524 kB)
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Using cached protobuf-3.20.0-py2.py3-none-any.whl.metadata (720 bytes)
  Using cached gradio-3.40.0-py3-none-any.whl.metadata (17 kB)
  Using cached aiofiles-23.2.1-py3-none-any.whl.metadata (9.7 kB)
  Using cached fastapi-0.115.12-py3-none-any.whl.metadata (27 kB)
  Using cached ffmpy-0.5.0-py3-none-any.whl.metadata (3.0 kB)
  Using cached gradio_client-1.10.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached importlib_resources-6.5.2-py3-none-any.whl.metadata (3.9 kB)
  Using cached orjson-3.10.18-cp312-cp312-win_amd64.whl.metadata (43 kB)
  Using cached pydub-0.25.1-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached python_multipart-0.0.20-py3-none-any.whl.metadata (1.8 kB)
  Using cached semantic_version-2.10.0-py2.py3-none-any.whl.metadata (9.7 kB)
  Using cached uvicorn-0.34.2-py3-none-any.whl.metadata (6.5 kB)
  Using cach

  error: subprocess-exited-with-error
  
  python setup.py bdist_wheel did not run successfully.
  exit code: 1
  
  [13 lines of output]
  C:\Users\Admin\anaconda3\Lib\site-packages\setuptools\_distutils\dist.py:261: UserWarning: Unknown distribution option: 'test_suite'
    warnings.warn(msg)
  running bdist_wheel
  running build
  running build_py
  creating build\lib.win-amd64-cpython-312\sentencepiece
  copying src\sentencepiece/__init__.py -> build\lib.win-amd64-cpython-312\sentencepiece
  copying src\sentencepiece/_version.py -> build\lib.win-amd64-cpython-312\sentencepiece
  copying src\sentencepiece/sentencepiece_model_pb2.py -> build\lib.win-amd64-cpython-312\sentencepiece
  copying src\sentencepiece/sentencepiece_pb2.py -> build\lib.win-amd64-cpython-312\sentencepiece
  running build_ext
  building 'sentencepiece._sentencepiece' extension
  error: Microsoft Visual C++ 14.0 or greater is required. Get it with "Microsoft C++ Build Tools": https://visualstudio.microsoft.com/vis

In [5]:
nvidia-smi

NameError: name 'nvidia' is not defined

## 패키지 불러오기

In [ ]:
from PIL import Image
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
import matplotlib.pyplot as plt

## 예제 이미지 불러오기

In [ ]:
# Load image
image = Image.open("examples/Hello.png").convert("RGB")

plt.figure(figsize=(10, 10))
plt.imshow(image)
plt.axis("on")
plt.show()

## 사전 학습 모델 불러오기

huggingface의 transformers에 구현되어 있는 TrOCR 모델을 불러옵니다. transfomers 라이브러리에서 pre-train 된 TrOCR 모델을 제공합니다.

transfomers의 라이브러리에서 두 가지 class를 불러옵니다.
- TrOCRProcessor: 모델에 입력할 데이터의 전처리와 출력된 모델 결과물의 후처리를 수행합니다.
- VisionEncoderDecoderModel: 실제 모델이며 전처리된 입력을 받으면 결과물을 출력합니다.

In [ ]:
print("[INFO] Load pretrained TrOCRProcessor")
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
print("[INFO] Load pretrained VisionEncoderDecoderModel")
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")

## TrOCR 모델로 이미지에서 텍스트 추론하기

TrOCRProcessor와 VisionEncoderDecoderModel의 내부 함수를 이용해 전처리, 추론, 후처리를 수행하여 원하는 결과물을 얻습니다.

In [ ]:
# Preprocess
pixel_values = processor(images=image, return_tensors="pt").pixel_values
# Inference
token_ids = model.generate(pixel_values)
# Postprocess
text_from_image = processor.batch_decode(token_ids, skip_special_tokens=True)[0]

In [ ]:
text_from_image